# Ajuste científico con SciPy y Recorda

Ensayo controlado de `uibcdf/recorda-lab#4`. Usamos `scipy.optimize.curve_fit` sin
modificar SciPy. Los datos ficticios y todos los ejes son **adimensionales**.
El alias decorado pertenece a este laboratorio; las llamadas siguen siendo normales.

Selecciona el entorno científico Python 3.14 definido en
`devtools/conda-envs/scientific_env.yaml`, con ambos checkouts instalados. Cada
ejecución completa crea un directorio nuevo. Los archivos NumPy y el modelo se
conservan fuera del journal. El registro cubre el ajuste declarado, no cada evaluación
interna del modelo ni la creación de datos o figuras.

In [ ]:
import sys
from pathlib import Path
from uuid import uuid4

import numpy as np
import recorda

lab_root = Path.cwd()
if lab_root.name == "notebooks":
    lab_root = lab_root.parent
assert (lab_root / "experiments").is_dir()
sys.path.insert(0, str(lab_root / "experiments"))

run_dir = lab_root / "artifacts" / f"scipy-notebook-{uuid4().hex}"
run_dir.mkdir(parents=True, exist_ok=False)
print(sys.executable)
print(run_dir)

In [ ]:
from scipy_consumer import FitArtifacts, affine, recorded_curve_fit

artifacts = FitArtifacts(run_dir / "native")
xdata = np.linspace(-2.0, 3.0, 11, dtype=np.float64)
noise = np.array([-0.1, 0.08, 0.04, -0.02, 0.06, -0.07, 0.03, -0.01, 0.09, -0.06, 0.01])
ydata = affine(xdata, 2.5, -0.7) + noise
p0 = np.array([1.0, 0.0])
unbounded = np.array([-np.inf, np.inf])
bounds = np.array([[-10.0, -10.0], [10.0, 10.0]])
for name, value in {
    "xdata": xdata,
    "ydata": ydata,
    "p0": p0,
    "unbounded": unbounded,
    "bounds": bounds,
}.items():
    artifacts.register_input(name, value)

ordinary = recorded_curve_fit(
    affine, xdata, ydata, p0=p0, bounds=unbounded, method="lm", maxfev=1000
)
assert not list(run_dir.glob("*.jsonl")) and not artifacts.results

## Un intento fallido y un ajuste correcto

Primero imponemos un límite de evaluaciones insuficiente. El reintento conserva
los mismos datos y modelo, cambiando esa configuración. El fallo queda visible.

In [ ]:
recorda.start(
    "fit_retry",
    path=run_dir / "retry.jsonl",
    reference_adapters=artifacts.adapters,
    gaps=["data generation, residual analysis and optimizer internals are unobserved"],
)

In [ ]:
try:
    recorded_curve_fit(affine, xdata, ydata, p0=p0, bounds=unbounded, method="lm", maxfev=1)
except RuntimeError as error:
    print(type(error).__name__)
else:
    raise AssertionError("El optimizador debía agotar el límite")

fitted = recorded_curve_fit(affine, xdata, ydata, p0=p0, bounds=unbounded, method="lm", maxfev=1000)
assert artifacts.results[0][0] is fitted
for direct, observed in zip(ordinary, fitted, strict=True):
    np.testing.assert_array_equal(direct, observed)
print("Pendiente e intercepto:", fitted[0])
print("Covarianza:\n", fitted[1])

In [ ]:
retry_record = recorda.stop()
assert retry_record.status == "failed"
assert [operation["status"] for operation in retry_record.operations] == ["failed", "succeeded"]
oracle = np.linalg.lstsq(np.column_stack([xdata, np.ones_like(xdata)]), ydata, rcond=None)[0]
np.testing.assert_allclose(fitted[0], oracle, rtol=1e-7, atol=1e-7)
print("Estado de la sesión:", retry_record.status)

## Comparación con otro método

El segundo ajuste usa `trf` y límites explícitos. El resultado nativo permanece
en SciPy/NumPy; la referencia del journal apunta a su manifiesto de archivos.

In [ ]:
recorda.start(
    "fit_comparison", path=run_dir / "comparison.jsonl", reference_adapters=artifacts.adapters
)
alternative = recorded_curve_fit(
    affine, xdata, ydata, p0=p0, bounds=bounds, method="trf", max_nfev=1000
)
comparison_record = recorda.stop()
assert comparison_record.status == "succeeded"
np.testing.assert_allclose(alternative[0], oracle, rtol=1e-7, atol=1e-7)
artifacts.write_index()

## Datos, ajuste y residuales

Esta figura es análisis del consumidor y queda fuera de las operaciones registradas.
Se exporta como SVG. El journal conserva las referencias a los datos y al ajuste.

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import SVG, display

fig, axes = plt.subplots(
    2, 1, figsize=(7, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]}, layout="constrained"
)
axes[0].scatter(xdata, ydata, label="Datos ficticios")
axes[0].plot(xdata, affine(xdata, *fitted[0]), label="Ajuste SciPy")
axes[0].set_ylabel("y (adimensional)")
axes[0].legend()
axes[1].scatter(xdata, ydata - affine(xdata, *fitted[0]))
axes[1].axhline(0, color="grey", linewidth=1)
axes[1].set_ylabel("Residual")
axes[1].set_xlabel("x (adimensional)")
figure_path = run_dir / "fit.svg"
fig.savefig(figure_path)
plt.close(fig)
display(SVG(filename=str(figure_path)))

## Qué puede reconstruir el registro

El inspector propio del ensayo lee el journal y comprueba las referencias/recibos
de archivos con Recorda y la biblioteca estándar, sin importar SciPy ni NumPy.
No ejecuta el modelo ni promete replay o integridad criptográfica del registro.

In [ ]:
from inspect_scipy_trial import inspect_trial

inspection = inspect_trial(run_dir)
for name, record in inspection["records"].items():
    print(name, record["status"])
    for operation in record["operations"]:
        print(
            operation["method"],
            operation["solver_options"],
            operation["status"],
            operation["implementation"]["version"],
        )
assert inspection["records"]["retry"]["operations"][0]["result_reference"] is None
assert inspection["records"]["retry"]["operations"][1]["result_reference"]["kind"] == "reference"

Los adaptadores son deliberadamente estrechos: entradas registradas, este modelo,
opciones numéricas permitidas y el resultado nativo de dos arrays de este ajuste.
Objetos desconocidos y `full_output=True` quedan omitidos de forma explícita.
La persistencia de archivos corresponde al consumidor; una referencia sola no
garantiza que sus bytes sigan disponibles. La prueba no califica todo SciPy ni todos
los trabajos de investigación. El siguiente ensayo semántico será Sabueso.